# Long-Form Music Generation with Magenta RT

This notebook demonstrates how to generate a longer piece of music (around 3 minutes) with distinct sections based on different text prompts using the Magenta RT library.

We will:
1. Install necessary dependencies.
2. Initialize the Magenta RT model.
3. Define a sequence of prompts and their desired durations.
4. Generate audio for each prompt and concatenate the results.
5. Play the final long-form musical piece.

## 1. Setup: Install Dependencies and Import Libraries

In [ ]:
# @title Install Magenta RT and dependencies (may take ~5 minutes)
# @markdown **Important:** Make sure you are using a TPU runtime for this notebook (`Runtime > Change runtime type > TPU`). Colab may prompt you to restart the session after installation. If so, wait for the cell to finish, then restart and run subsequent cells.

# Clone the repository (if not already done or running in a fresh environment)
!git clone https://github.com/magenta/magenta-realtime.git

# Uninstall existing TensorFlow versions to avoid conflicts and install specific nightly versions
_all_tf = 'tensorflow tf-nightly tensorflow-cpu tf-nightly-cpu tensorflow-tpu tf-nightly-tpu tensorflow-hub tf-hub-nightly tensorflow-text tensorflow-text-nightly'
_nightly_tf = 'tf-nightly tensorflow-text-nightly tf-hub-nightly'

# Install Magenta RT with TPU support
!pip install -e magenta-realtime/[tpu] && pip uninstall -y {_all_tf} && pip install {_nightly_tf}

In [ ]:
# @title Import necessary libraries and initialize the model (may take ~5 minutes after restart)
# @markdown If you restarted the session after the previous cell, run this cell to import libraries and load the model.

import numpy as np
from IPython.display import Audio, display

from magenta_rt import system
from magenta_rt import audio

print("Initializing MagentaRT model...")
try:
    # Attempt to initialize if running after a fresh pip install in the same session
    MRT = system.MagentaRT(
        tag="large", device="tpu:v2-8", skip_cache=True, lazy=False
    )
    print("MagentaRT model initialized successfully.")
except NameError:
    # This block might be hit if the notebook is run cell-by-cell after a restart
    # or if the pip install was in a previous session.
    print("MagentaRT class not found, likely due to session restart. This is expected.")
    print("Please ensure the previous cell (installation) completed and you've restarted if prompted.")
    print("The model will be loaded in the next code cell that explicitly calls system.MagentaRT().")
except Exception as e:
    print(f"An error occurred during initialization: {e}")
    print("Please ensure you are on a TPU runtime and have restarted the session if prompted after installation.")

If the cell above printed messages about `MagentaRT class not found` or `NameError`, it's often because Colab requires a session restart after `pip install`. After restarting (`Runtime > Restart session`), run the cell above again. It should then print `MagentaRT model initialized successfully.` (or attempt to initialize it). If it still fails, re-run the installation cell first, restart, then run the initialization cell.

Let's explicitly initialize the model here again to be sure, especially if the above cell indicated a `NameError`.

In [ ]:
# @title Ensure Model is Loaded
# @markdown Run this cell to explicitly load or confirm the model is loaded.
try:
    if 'MRT' not in globals() or MRT is None:
        print("MRT model not found or not initialized. Attempting to load now...")
        MRT = system.MagentaRT(
            tag="large", device="tpu:v2-8", skip_cache=True, lazy=False
        )
        print("MagentaRT model loaded successfully.")
    else:
        print("MagentaRT model is already loaded.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Please ensure installation completed, you've restarted the session if prompted, and are on a TPU runtime.")

## 2. Define Generation Parameters

Here, we'll specify the prompts and their durations to structure our long-form musical piece. Each prompt will define a section of the music.

In [ ]:
# @title Define prompts and their durations (in seconds)
PROMPT_SECTIONS = [
    {"prompt": "Epic cinematic orchestral score, dramatic strings, powerful brass", "duration_seconds": 60},
    {"prompt": "Calm peaceful piano solo, melancholic melody", "duration_seconds": 45},
    {"prompt": "Upbeat funky jazz fusion, groovy bassline, saxophone lead", "duration_seconds": 60},
    {"prompt": "Ambient electronic soundscape, atmospheric pads, gentle rhythm", "duration_seconds": 45},
    {"prompt": "Energetic rock anthem, distorted guitars, driving drums", "duration_seconds": 30}
]

TOTAL_DURATION_SECONDS = sum(section['duration_seconds'] for section in PROMPT_SECTIONS)
print(f"Total planned duration: {TOTAL_DURATION_SECONDS // 60} minutes and {TOTAL_DURATION_SECONDS % 60} seconds")

# Model and generation settings
CHUNK_LENGTH_SECONDS = MRT.config.chunk_length_seconds # Typically 2 seconds
SAMPLE_RATE = MRT.sample_rate # Typically 48000 Hz
CROSSFADE_SECONDS = MRT.config.crossfade_length # Typically 0.04 seconds (40ms)

print(f"Chunk length: {CHUNK_LENGTH_SECONDS}s, Sample rate: {SAMPLE_RATE}Hz, Crossfade: {CROSSFADE_SECONDS}s")

## 3. Implement the Generation Loop

Now we'll loop through our defined sections. For each section:
1. We get the style embedding for the prompt.
2. We calculate how many audio chunks are needed for its duration.
3. We generate these chunks sequentially, maintaining the model's state between chunks for coherence.
4. All generated chunks are collected.

In [ ]:
# @title Generate audio section by section
import time
import math

all_generated_chunks = []
current_model_state = None # Initialize model state

total_sections = len(PROMPT_SECTIONS)
for i, section in enumerate(PROMPT_SECTIONS):
    prompt = section["prompt"]
    duration_seconds = section["duration_seconds"]
    print(f"\nGenerating section {i+1}/{total_sections}: '{prompt}' for {duration_seconds}s...")

    # Get style embedding for the current prompt
    style_embedding = MRT.embed_style(prompt)

    # Calculate the number of chunks for this section
    num_chunks_for_section = math.ceil(duration_seconds / CHUNK_LENGTH_SECONDS)

    section_chunks = []
    start_time = time.time()
    for chunk_idx in range(num_chunks_for_section):
        # Generate one chunk
        # You can adjust generation parameters like temperature, top_k, guidance_weight here if desired
        generated_audio_chunk, current_model_state = MRT.generate_chunk(
            state=current_model_state,
            style=style_embedding,
            seed=i * 1000 + chunk_idx, # Optional: for reproducibility, vary seed per chunk
            temperature=1.0, # Default is 1.0, adjust for more/less randomness
            top_k=0, # Default is 0 (no top-k filtering), adjust if needed
            guidance_weight=3.0 # Default is 3.0, adjust for stronger/weaker prompt adherence
        )
        section_chunks.append(generated_audio_chunk)
        print(f"  Generated chunk {chunk_idx + 1}/{num_chunks_for_section} for section '{prompt}'", end='\r')
    
    all_generated_chunks.extend(section_chunks)
    end_time = time.time()
    print(f"\nSection '{prompt}' ({duration_seconds}s) generated in {end_time - start_time:.2f}s.")

print("\nAll sections generated!")

## 4. Concatenate and Display Audio

Finally, we'll take all the generated audio chunks and concatenate them into a single audio stream. We use the model's configured crossfade duration to ensure smooth transitions between chunks.

In [ ]:
# @title Concatenate all chunks and display the final audio
if all_generated_chunks:
    print(f"\nConcatenating {len(all_generated_chunks)} chunks...")
    final_audio = audio.concatenate(
        all_generated_chunks,
        crossfade_time=CROSSFADE_SECONDS,
    )
    print("Concatenation complete.")

    # Display the audio player
    print(f"Playing final audio ({final_audio.duration_seconds:.2f}s total duration):")
    display(Audio(final_audio.samples.T, rate=final_audio.sample_rate))
else:
    print("No audio chunks were generated. Please check previous steps.")